# FreeFine Final Geometry — Step 0: GeoBench-2D Audit & Router Freeze

**Purpose:** before the full 5,677-sample end-to-end final runs, verify the exact GeoBench-2D population and derive/validate a deployable resize-severity rule from the requested geometric transform (not from oracle benchmark labels).

This notebook performs **no model inference and needs no GPU**. It does not tune on generated-image metrics.

Expected GeoBench-2D population used in the thesis reproduction: **5,677 edits = 1,439 move + 1,603 rotate + 2,635 resize**.

The FreeFine paper defines resize intensity ranges as:
- easy: enlarge 1.1–1.3, shrink 0.8–0.9
- medium: enlarge 1.3–1.5, shrink 0.6–0.8
- hard: enlarge 1.5–3.0, shrink 0.4–0.6

The candidate deployable rule to audit is therefore `severe_resize := (scale >= 1.5) or (scale <= 0.6)`.


In [1]:
import os, glob, csv, json, math, re, hashlib
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image

print("Kaggle inputs:")
for p in sorted(glob.glob("/kaggle/input/*")):
    print(" -", p)

def one(pattern, desc, required=True):
    xs = glob.glob(pattern, recursive=True)
    xs = [x for x in xs if os.path.exists(x)]
    if not xs:
        if required:
            raise FileNotFoundError(f"Missing {desc}. Pattern: {pattern}")
        return None
    # Prefer shortest path to avoid accidental nested copies.
    xs = sorted(xs, key=lambda x: (len(x), x))
    return xs[0]

# Primary GeoBench assets.
geo_roots = [
    p for p in glob.glob("/kaggle/input/**/Geo-Bench-2D", recursive=True)
    if os.path.isdir(p)
]
if not geo_roots:
    raise FileNotFoundError(
        "Could not find a Geo-Bench-2D directory under /kaggle/input. "
        "Attach the original GeoBench input used for your FreeFine runs."
    )
GEO2D = sorted(geo_roots, key=lambda x: (len(x), x))[0]
META = one("/kaggle/input/**/sample_metadata.csv", "sample_metadata.csv")
ANN  = one("/kaggle/input/**/annotation_2d.json", "annotation_2d.json", required=False)

print("\nResolved:")
print("GEO2D =", GEO2D)
print("META  =", META)
print("ANN   =", ANN)

df = pd.read_csv(META, dtype=str)
print("\nmetadata columns:", list(df.columns))
print("raw rows:", len(df))

required_cols = {"da_n", "ins_id", "case_id", "edit_type", "difficulty"}
missing = required_cols - set(df.columns)
assert not missing, f"sample_metadata.csv missing required columns: {missing}"

for c in ["edit_type", "difficulty"]:
    df[c] = df[c].astype(str).str.strip().str.lower()

# Keep the general 2D edits only.
df2 = df[df["edit_type"].isin(["move", "rotate", "resize"])].copy()
print("\n2D edit counts:")
print(df2["edit_type"].value_counts().sort_index())
print("total =", len(df2))

EXPECTED = {"move": 1439, "rotate": 1603, "resize": 2635}
got = df2["edit_type"].value_counts().to_dict()
assert len(df2) == 5677, f"Expected 5677 general 2D edits, found {len(df2)}"
assert got == EXPECTED, f"Unexpected edit counts. expected={EXPECTED}, got={got}"

print("\nDifficulty x edit type:")
print(pd.crosstab(df2["edit_type"], df2["difficulty"]))

# Stable manifest hash, useful later to prove all final pipelines use identical samples.
keys = sorted(
    f"{r.da_n}|{r.ins_id}|{r.case_id}|{r.edit_type}|{r.difficulty}"
    for r in df2.itertuples()
)
manifest_sha = hashlib.sha256("\n".join(keys).encode()).hexdigest()
print("\nGeoBench-2D manifest SHA256:", manifest_sha)

df2.to_csv("/kaggle/working/geobench_2d_manifest_5677.csv", index=False)
print("saved /kaggle/working/geobench_2d_manifest_5677.csv")


Kaggle inputs:
 - /kaggle/input/datasets
 - /kaggle/input/geobench2d-metrics-subset

Resolved:
GEO2D = /kaggle/input/geobench2d-metrics-subset/geobench_metrics/Geo-Bench-2D
META  = /kaggle/input/datasets/georgiostzamouranis/freefine-sample-metadata/sample_metadata.csv
ANN   = /kaggle/input/geobench2d-metrics-subset/geobench_metrics/annotation_2d.json

metadata columns: ['da_n', 'ins_id', 'case_id', 'edit_type', 'norm_translation', 'rotation_deg', 'scale_change', 'mask_area_ratio', 'overlap_iou', 'gen_rel', 'difficulty']
raw rows: 5677

2D edit counts:
edit_type
move      1439
resize    2635
rotate    1603
Name: count, dtype: int64
total = 5677

Difficulty x edit type:
difficulty  easy  hard  medium
edit_type                     
move         484   413     542
resize       974   800     861
rotate       434   680     489

GeoBench-2D manifest SHA256: 4a72cd34c1e4dadf3ea8dc88cdd36b5a7a8d3c11ca6b424556fb7170f9f33f02
saved /kaggle/working/geobench_2d_manifest_5677.csv


In [2]:
# --- Recover a numeric resize scale without depending on benchmark difficulty labels. ---
#
# Preferred order:
#   1) explicit numeric metadata column with a scale-like name;
#   2) explicit numeric leaf in annotation_2d.json with a scale-like key path;
#   3) target/source mask area ratio: s ~= sqrt(area(target)/area(source)).
#
# The notebook prints exactly which source was used.

resize = df2[df2["edit_type"] == "resize"].copy()

def numeric_series(s):
    x = pd.to_numeric(s, errors="coerce")
    return x

# 1) metadata candidate columns
meta_candidates = []
for c in df2.columns:
    lc = c.lower()
    if any(k in lc for k in ["scale", "resize", "ratio", "factor", "zoom"]):
        vals = numeric_series(resize[c])
        frac = vals.notna().mean()
        if frac > 0.90:
            meta_candidates.append((c, frac, vals.min(), vals.max()))

print("Explicit numeric metadata candidates:")
print(meta_candidates if meta_candidates else "  none")

scale_source = None
scale_values = None

# choose the most obviously scale-like metadata column, if present
preferred = [x for x in meta_candidates if "scale" in x[0].lower()]
if preferred:
    col = preferred[0][0]
    scale_values = numeric_series(resize[col]).astype(float)
    scale_source = f"sample_metadata.csv::{col}"

# 2) annotation candidate key paths, only if metadata did not directly expose scale.
ann = json.load(open(ANN)) if ANN else None

def get_case_obj(row):
    if ann is None:
        return None
    try:
        return ann[str(row["da_n"])]["instances"][str(row["ins_id"])][str(row["case_id"])]
    except Exception:
        # tolerate integer-like JSON keys / dataframe strings
        try:
            return ann[row["da_n"]]["instances"][row["ins_id"]][row["case_id"]]
        except Exception:
            return None

def numeric_leaves(obj, prefix=""):
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            out.extend(numeric_leaves(v, p))
    elif isinstance(obj, list):
        # We do not automatically interpret arbitrary matrices/lists as scale.
        # Scalar singleton lists are okay.
        if len(obj) == 1:
            out.extend(numeric_leaves(obj[0], prefix))
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        out.append((prefix, float(obj)))
    return out

if scale_values is None and ann is not None:
    path_vals = defaultdict(list)
    # sample enough to identify stable key paths
    for _, r in resize.head(500).iterrows():
        obj = get_case_obj(r)
        if obj is None:
            continue
        for p, v in numeric_leaves(obj):
            path_vals[p].append(v)

    cand = []
    for p, vals in path_vals.items():
        lp = p.lower()
        if len(vals) >= 300 and any(k in lp for k in ["scale", "resize", "ratio", "factor", "zoom"]):
            a = np.array(vals, dtype=float)
            cand.append((p, len(vals), float(np.nanmin(a)), float(np.nanmax(a))))
    print("\nAnnotation numeric scale-like candidates:")
    print(cand[:30] if cand else "  none")

    # exact extraction from the best candidate if one is clearly named scale
    strong = [x for x in cand if "scale" in x[0].lower()]
    if strong:
        pth = strong[0][0]

        def read_path(obj, path):
            cur = obj
            for part in path.split("."):
                cur = cur[part]
            if isinstance(cur, list) and len(cur) == 1:
                cur = cur[0]
            return float(cur)

        vals = []
        ok = True
        for _, r in resize.iterrows():
            obj = get_case_obj(r)
            try:
                vals.append(read_path(obj, pth))
            except Exception:
                ok = False
                break
        if ok:
            scale_values = pd.Series(vals, index=resize.index, dtype=float)
            scale_source = f"annotation_2d.json::{pth}"

# 3) Mask area fallback, robust to unknown annotation schema.
def find_mask_path(root, row, kind):
    d, i, c = str(row["da_n"]), str(row["ins_id"]), str(row["case_id"])
    base = os.path.join(root, kind)

    candidates = [
        os.path.join(base, d, i, f"{c}.png"),
        os.path.join(base, d, i, c, "mask.png"),
        os.path.join(base, d, i, c, "target_mask.png"),
        os.path.join(base, d, i, c, "source_mask.png"),
        os.path.join(base, d, i, "mask.png"),
        os.path.join(base, d, i, "source_mask.png"),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p

    # constrained glob fallback
    pats = [
        os.path.join(base, d, i, f"*{c}*.png"),
        os.path.join(base, d, i, c, "*.png"),
        os.path.join(base, d, i, "*.png"),
    ]
    xs = []
    for pat in pats:
        xs.extend(glob.glob(pat))
    xs = sorted(set(x for x in xs if os.path.isfile(x)))
    if not xs:
        return None
    # for source mask, a single instance-level file may be shared across cases
    if kind == "source_mask" and len(xs) == 1:
        return xs[0]
    # prefer exact case token if possible
    exact = [x for x in xs if os.path.splitext(os.path.basename(x))[0] == c]
    return exact[0] if exact else xs[0]

def mask_area(path):
    a = np.array(Image.open(path).convert("L"))
    return float((a > 127).sum())

if scale_values is None:
    vals = []
    failures = []
    for idx, r in resize.iterrows():
        sp = find_mask_path(GEO2D, r, "source_mask")
        tp = find_mask_path(GEO2D, r, "target_mask")
        if sp is None or tp is None:
            failures.append((r["da_n"], r["ins_id"], r["case_id"], sp, tp))
            vals.append(np.nan)
            continue
        sa, ta = mask_area(sp), mask_area(tp)
        vals.append(math.sqrt(ta / sa) if sa > 0 else np.nan)

    scale_values = pd.Series(vals, index=resize.index, dtype=float)
    coverage = scale_values.notna().mean()
    print("\nMask-ratio scale coverage:", coverage)
    if coverage < 0.99:
        print("first mask lookup failures:", failures[:20])
        raise RuntimeError(
            "Could not recover resize scale for >=99% of samples. "
            "Inspect the printed mask paths / annotation candidates before proceeding."
        )
    scale_source = "sqrt(target_mask_area/source_mask_area)"

resize["scale"] = scale_values
print("\nSCALE SOURCE:", scale_source)
print(resize["scale"].describe())


Explicit numeric metadata candidates:
[('scale_change', np.float64(1.0), 0.1907421171081432, 2.1158771720529286), ('mask_area_ratio', np.float64(1.0), 0.00749755859375, 0.75653564453125)]

SCALE SOURCE: sample_metadata.csv::scale_change
count    2635.000000
mean        0.753784
std         0.457375
min         0.190742
25%         0.370404
50%         0.623690
75%         1.074558
max         2.115877
Name: scale, dtype: float64


In [3]:
# --- Audit against the paper-defined resize intensity bands. ---
#
# IMPORTANT:
# We are not optimizing a threshold from model metrics here.
# We are checking whether the known requested scale itself reproduces the benchmark's
# easy/medium/hard organization.

stats = resize.groupby("difficulty")["scale"].agg(["count","min","max","mean","median"]).reindex(
    ["easy","medium","hard"]
)
print("Scale statistics by benchmark label:")
display(stats)

# Paper table:
# easy   enlarge 1.1-1.3 ; shrink 0.8-0.9
# medium enlarge 1.3-1.5 ; shrink 0.6-0.8
# hard   enlarge 1.5-3.0 ; shrink 0.4-0.6
#
# Candidate deployment rule:
# severe iff requested scale is at/above 1.5x enlargement or at/below 0.6x shrink.
ENLARGE_SEVERE = 1.5
SHRINK_SEVERE = 0.6

resize["router_severe"] = (resize["scale"] >= ENLARGE_SEVERE) | (resize["scale"] <= SHRINK_SEVERE)
resize["benchmark_hard"] = resize["difficulty"].eq("hard")

ct = pd.crosstab(
    resize["benchmark_hard"],
    resize["router_severe"],
    rownames=["benchmark_hard"],
    colnames=["router_severe"],
    margins=True,
)
print("\nHard-label vs affine/mask-derived severity:")
display(ct)

agree = (resize["benchmark_hard"] == resize["router_severe"]).mean()
print("agreement =", agree)

# Boundary diagnostics are critical because the paper's printed intervals meet at 0.6 and 1.5.
eps = 0.03 if scale_source.startswith("sqrt(") else 1e-8
boundary = resize[
    (resize["scale"].sub(1.5).abs() <= eps) |
    (resize["scale"].sub(0.6).abs() <= eps)
][["da_n","ins_id","case_id","difficulty","scale"]].sort_values("scale")
print(f"\nBoundary-near samples (eps={eps}):", len(boundary))
display(boundary.head(100))

mismatch = resize[resize["benchmark_hard"] != resize["router_severe"]][
    ["da_n","ins_id","case_id","difficulty","scale"]
].sort_values("scale")
print("\nMismatches:", len(mismatch))
display(mismatch.head(100))

# Do not silently freeze a bad rule.
# Exact metadata/annotation scale should normally be nearly perfect.
# Mask-ratio fallback can have small rasterization drift around the exact boundaries,
# so mismatches near 0.6/1.5 are reviewed rather than blindly accepted.
if len(mismatch) == 0:
    audit_status = "PASS_EXACT"
elif scale_source.startswith("sqrt("):
    far = mismatch[
        (mismatch["scale"].sub(1.5).abs() > 0.05) &
        (mismatch["scale"].sub(0.6).abs() > 0.05)
    ]
    audit_status = "PASS_MASK_RATIO_WITH_BOUNDARY_REVIEW" if len(far) == 0 else "FAIL"
else:
    audit_status = "FAIL"

print("\nAUDIT STATUS:", audit_status)
if audit_status == "FAIL":
    raise RuntimeError(
        "Severity rule does not reproduce the benchmark's resize organization well enough. "
        "Do NOT start the full final benchmark yet."
    )


Scale statistics by benchmark label:


,count,min,max,mean,median
difficulty,,,,,
easy,974,0.190742,0.827952,0.401461,0.365253
medium,861,0.190742,1.351120,0.666194,0.661670
hard,800,0.191928,2.115877,1.277008,1.346182



Hard-label vs affine/mask-derived severity:


router_severe,False,True,All
benchmark_hard,,,
False,665,1170,1835
True,467,333,800
All,1132,1503,2635


agreement = 0.37874762808349144

Boundary-near samples (eps=1e-08): 0


,da_n,ins_id,case_id,difficulty,scale



Mismatches: 1637


,da_n,ins_id,case_id,difficulty,scale
1915,216,0,6,easy,0.190742
2746,319,0,6,easy,0.190742
2736,318,0,6,easy,0.190742
1079,127,0,4,medium,0.190742
1014,118,0,6,easy,0.191553
...,...,...,...,...,...
5638,605,0,6,easy,0.225993
5649,606,0,6,easy,0.225993
4813,527,0,6,easy,0.225993
127,16,0,6,easy,0.225993



AUDIT STATUS: FAIL


RuntimeError: Severity rule does not reproduce the benchmark's resize organization well enough. Do NOT start the full final benchmark yet.

In [ ]:
# --- Freeze artifacts for the subsequent full end-to-end runs. ---
#
# We save both the benchmark audit and the router configuration.
# The final inference pack will read this JSON verbatim; it must not be edited after
# looking at final 5,677-sample model metrics.

router_cfg = {
    "version": "SGR_router_pre_full_v1",
    "resize_scale_source": scale_source,
    "resize_severe_rule": {
        "enlarge": "scale >= 1.5",
        "shrink": "scale <= 0.6",
        "enlarge_threshold": 1.5,
        "shrink_threshold": 0.6,
    },
    "branches_common": {
        "move": "RING4_MOVE_POST",
        "rotate": "FREEFINE_BASELINE",
        "resize_nonsevere": "RING8_GLOBAL_POST",
    },
    "pipeline_A_resize_severe": "EPSREC_PROMPT_ALL",
    "pipeline_B_resize_severe": "MIDHF_EPSREC_PROMPT_ALL",
    "geobench_2d_n": int(len(df2)),
    "geobench_2d_manifest_sha256": manifest_sha,
    "audit_status": audit_status,
    "note": (
        "Difficulty labels are used only for audit/reporting. "
        "Final branch selection must use requested scale / affine-derived scale, never difficulty."
    ),
}

with open("/kaggle/working/final_geometry_router_config.json", "w") as f:
    json.dump(router_cfg, f, indent=2)

resize.to_csv("/kaggle/working/resize_severity_audit_2635.csv", index=False)

summary = {
    "manifest_sha256": manifest_sha,
    "counts": got,
    "resize_scale_source": scale_source,
    "resize_scale_stats": stats.reset_index().to_dict(orient="records"),
    "agreement": float(agree),
    "mismatch_count": int(len(mismatch)),
    "boundary_near_count": int(len(boundary)),
    "audit_status": audit_status,
}
with open("/kaggle/working/geobench_router_audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(router_cfg, indent=2))
print("\nSaved:")
print(" /kaggle/working/final_geometry_router_config.json")
print(" /kaggle/working/geobench_router_audit_summary.json")
print(" /kaggle/working/geobench_2d_manifest_5677.csv")
print(" /kaggle/working/resize_severity_audit_2635.csv")
print("\n✓ STEP 0 COMPLETE — safe to build the full end-to-end final run pack")
